In [1]:
import subprocess, sys
from pathlib import Path

whl = list(Path("/kaggle/input").rglob("onnxruntime*.whl"))[0]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(whl)], check=True)
print("✅ ONNX installed")

✅ ONNX installed


In [2]:
# ====================== 1. 导入 ======================
import os, re, gc
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import onnxruntime as ort
from pathlib import Path
from tqdm import tqdm
from scipy.ndimage import gaussian_filter1d

# ====================== 2. 路径探测 ======================
BASE = Path("/kaggle/input/competitions/birdclef-2026")
if not BASE.exists():
    BASE = Path("/kaggle/input/birdclef-2026")

perch_onnx_candidates = list(Path("/kaggle/input").rglob("perch_v2.onnx"))
if not perch_onnx_candidates:
    raise FileNotFoundError("找不到 perch_v2.onnx")
PERCH_ONNX = perch_onnx_candidates[0]

sed_paths = sorted(Path("/kaggle/input").rglob("sed_fold*.onnx"))
if not sed_paths:
    raise FileNotFoundError("找不到 sed_fold*.onnx")

print(f"BASE: {BASE}")
print(f"PERCH: {PERCH_ONNX}")
print(f"SED: {[p.name for p in sed_paths]}")

# ====================== 3. 常量 ======================
SR = 32000
WINDOW_SAMPLES = 160000
N_WINDOWS = 12
N_CLASSES = 234

# ====================== 4. 标签 ======================
sample_sub = pd.read_csv(BASE / "sample_submission.csv")
LABELS = [c for c in sample_sub.columns if c != "row_id"]
label_to_idx = {lab: i for i, lab in enumerate(LABELS)}

# ====================== 5. 辅助函数 ======================
def read_60s(path, sr_target=32000):
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr0 != sr_target:
        y = librosa.resample(y, orig_sr=sr0, target_sr=sr_target)
    target = 60 * sr_target
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]
    return y

def parse_meta(filename):
    m = re.search(r'_([A-Z]\d+)_\d{8}_(\d{6})\.ogg', filename)
    if m:
        return m.group(1), int(m.group(2)[:2])
    return "S00", 12

def hour_bucket(h):
    if h < 6:
        return "night"
    if h < 12:
        return "morning"
    if h < 18:
        return "day"
    return "evening"

# ====================== 6. Perch ONNX ======================
class PerchV2:
    def __init__(self, path, n_threads=4):
        so = ort.SessionOptions()
        so.intra_op_num_threads = n_threads
        self.sess = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
        self.in_name = self.sess.get_inputs()[0].name
        out_map = {o.name: i for i, o in enumerate(self.sess.get_outputs())}
        self.emb_idx = out_map["embedding"]
        self.lbl_idx = out_map["label"]

    def __call__(self, waveforms):
        outs = self.sess.run(None, {self.in_name: waveforms})
        return outs[self.emb_idx].astype(np.float32), outs[self.lbl_idx].astype(np.float32)

perch = PerchV2(PERCH_ONNX)

# ====================== 7. SED ONNX ======================
seds = []
for p in sed_paths:
    so = ort.SessionOptions()
    so.intra_op_num_threads = 2
    sess = ort.InferenceSession(str(p), so, providers=["CPUExecutionProvider"])
    seds.append({"sess": sess, "in_name": sess.get_inputs()[0].name})
print(f"SED models: {len(seds)}")

# ====================== 8. Perch → 234 映射 ======================
taxonomy = pd.read_csv(BASE / "taxonomy.csv")

# 自动探测 labels.csv
labels_candidates = [
    PERCH_ONNX.parent / "assets" / "labels.csv",
    PERCH_ONNX.parent / "labels.csv",
    PERCH_ONNX.parent.parent / "labels.csv",
]
bc_labels = None
for p in labels_candidates:
    if p.exists():
        bc_labels = pd.read_csv(p)
        break
if bc_labels is None:
    found = list(Path("/kaggle/input").rglob("labels.csv"))
    if found:
        bc_labels = pd.read_csv(found[0])
    else:
        raise FileNotFoundError("找不到 labels.csv")

NO_LABEL = len(bc_labels)
bc_labels = bc_labels.reset_index().rename(columns={"index": "bc_idx", bc_labels.columns[0]: "sci_name"})
merged = taxonomy.merge(bc_labels, left_on="scientific_name", right_on="sci_name", how="left")
merged["bc_idx"] = merged["bc_idx"].fillna(NO_LABEL).astype(int)
lbl2bc = dict(zip(merged["primary_label"].astype(str), merged["bc_idx"]))

BC_IDX = np.array([int(lbl2bc.get(l, NO_LABEL)) for l in LABELS], dtype=np.int32)
MAPPED = BC_IDX != NO_LABEL
MAPPED_POS = np.where(MAPPED)[0].astype(np.int32)
MAPPED_BC = BC_IDX[MAPPED].astype(np.int32)
UNMAPPED_POS = np.where(~MAPPED)[0].astype(np.int32)

print(f"Mapped: {MAPPED.sum()}/{N_CLASSES} | Unmapped: {len(UNMAPPED_POS)}")

# ====================== 9. Genus Proxy ======================
genus_to_bc = {}
for _, row in taxonomy.iterrows():
    sci = str(row["scientific_name"])
    genus = sci.split()[0] if " " in sci else sci
    lbl = str(row["primary_label"])
    if lbl in label_to_idx and label_to_idx[lbl] in MAPPED_POS:
        bc_idx = BC_IDX[label_to_idx[lbl]]
        if genus not in genus_to_bc:
            genus_to_bc[genus] = []
        genus_to_bc[genus].append(bc_idx)

proxy_map = {}
for pos in UNMAPPED_POS:
    lbl = LABELS[pos]
    sci = str(taxonomy[taxonomy["primary_label"] == lbl]["scientific_name"].values[0]) if len(taxonomy[taxonomy["primary_label"] == lbl]) > 0 else ""
    genus = sci.split()[0] if " " in sci else sci
    proxy_map[pos] = genus_to_bc.get(genus, MAPPED_BC.tolist())

print(f"Genus proxy: {len(proxy_map)} classes")

# ====================== 10. Site / Hour 先验 ======================
train_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")
train_labels["site"] = train_labels["filename"].apply(lambda x: parse_meta(x)[0])
train_labels["hour"] = train_labels["filename"].apply(lambda x: parse_meta(x)[1])
train_labels["hour_bucket"] = train_labels["hour"].apply(hour_bucket)

# 全局先验
all_labs = []
for txt in train_labels["primary_label"].fillna(""):
    all_labs.extend([t.strip() for t in str(txt).split(";") if t.strip()])
global_prior = np.zeros(N_CLASSES, dtype=np.float32)
for lab, cnt in pd.Series(all_labs).value_counts().items():
    if lab in label_to_idx:
        global_prior[label_to_idx[lab]] = cnt / len(all_labs)

# Site 先验
site_priors = {}
for site, group in train_labels.groupby("site"):
    labs = []
    for txt in group["primary_label"].fillna(""):
        labs.extend([t.strip() for t in str(txt).split(";") if t.strip()])
    if not labs:
        continue
    prior = np.zeros(N_CLASSES, dtype=np.float32)
    for lab, cnt in pd.Series(labs).value_counts().items():
        if lab in label_to_idx:
            prior[label_to_idx[lab]] = cnt / len(labs)
    site_priors[site] = prior

# Hour 先验
hour_priors = {}
for hb, group in train_labels.groupby("hour_bucket"):
    labs = []
    for txt in group["primary_label"].fillna(""):
        labs.extend([t.strip() for t in str(txt).split(";") if t.strip()])
    if not labs:
        continue
    prior = np.zeros(N_CLASSES, dtype=np.float32)
    for lab, cnt in pd.Series(labs).value_counts().items():
        if lab in label_to_idx:
            prior[label_to_idx[lab]] = cnt / len(labs)
    hour_priors[hb] = prior

print(f"Site priors: {len(site_priors)} | Hour: {list(hour_priors.keys())}")

# ====================== 11. 温度缩放 ======================
temperatures = np.ones(N_CLASSES, dtype=np.float32)
for i, lab in enumerate(LABELS):
    cls = str(taxonomy[taxonomy["primary_label"] == lab]["class_name"].values[0]) if len(taxonomy[taxonomy["primary_label"] == lab]) > 0 else "Aves"
    if cls in {"Amphibia", "Insecta"}:
        temperatures[i] = 0.90
    else:
        temperatures[i] = 1.15

# ====================== 12. SED Mel 预处理 ======================
N_MELS_SED = 256
N_FFT_SED = 2048
HOP_SED = 512

def sed_mel(path, sr=32000):
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr0 != sr:
        y = librosa.resample(y, orig_sr=sr0, target_sr=sr)
    target = 60 * sr
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]

    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=N_FFT_SED, hop_length=HOP_SED,
        n_mels=N_MELS_SED, fmin=20, fmax=16000, power=2.0)
    mel_db = librosa.power_to_db(mel, top_db=80)

    fpw = 5 * sr // HOP_SED + 1
    hf = 5 * sr // HOP_SED
    windows = []
    for i in range(N_WINDOWS):
        s = i * hf
        e = s + fpw
        if e > mel_db.shape[1]:
            c = np.zeros((N_MELS_SED, fpw), dtype=np.float32)
            a = mel_db.shape[1] - s
            if a > 0:
                c[:, :a] = mel_db[:, s:s + a]
        else:
            c = mel_db[:, s:e]
        m, sd = c.mean(), c.std() + 1e-6
        c = (c - m) / sd
        windows.append(c[None, :, :])
    return np.stack(windows).astype(np.float32)

# ====================== 13. 推理循环 ======================
test_dir = BASE / "test_soundscapes"
audio_exts = {".ogg", ".mp3", ".wav", ".flac", ".oga"}
test_files = sorted([f for f in test_dir.iterdir() if f.suffix.lower() in audio_exts])

DRYRUN = len(test_files) == 0
if DRYRUN:
    test_files = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:20]
    print(f"Dry-run: {len(test_files)} files")

all_probs = []
all_rids = []

pbar = tqdm(test_files, desc="Perch+SED", unit="file", ncols=80)
for f in pbar:
    try:
        # Perch
        y = read_60s(f).reshape(N_WINDOWS, WINDOW_SAMPLES).astype(np.float32)
        _, perch_logits = perch(y)

        perch_234 = np.zeros((N_WINDOWS, N_CLASSES), dtype=np.float32)
        perch_234[:, MAPPED_POS] = perch_logits[:, MAPPED_BC]
        for pos, bc_list in proxy_map.items():
            perch_234[:, pos] = perch_logits[:, bc_list].max(axis=1)

        perch_234 = perch_234 / temperatures[None, :]
        probs_perch = 1.0 / (1.0 + np.exp(-np.clip(perch_234, -50, 50)))

        # SED
        mel = sed_mel(f)
        sed_sum = np.zeros((N_WINDOWS, N_CLASSES), dtype=np.float32)
        for sed in seds:
            outs = sed["sess"].run(None, {sed["in_name"]: mel})
            clip = outs[0]
            frame = outs[1].max(axis=1) if len(outs) > 1 else clip
            p_clip = 1.0 / (1.0 + np.exp(-np.clip(clip, -50, 50)))
            p_frame = 1.0 / (1.0 + np.exp(-np.clip(frame, -50, 50)))
            sed_sum += 0.5 * p_clip + 0.5 * p_frame
        probs_sed = sed_sum / len(seds)

        # Rank-Average
        ra_p = pd.DataFrame(probs_perch).rank(axis=0, pct=True).to_numpy(np.float32)
        ra_s = pd.DataFrame(probs_sed).rank(axis=0, pct=True).to_numpy(np.float32)
        blended = 0.6 * ra_p + 0.4 * ra_s

        # Site/Hour 先验
        site, hour = parse_meta(f.name)
        hb = hour_bucket(hour)
        prior = global_prior.copy()
        if site in site_priors:
            prior = 0.6 * prior + 0.4 * site_priors[site]
        if hb in hour_priors:
            prior = 0.7 * prior + 0.3 * hour_priors[hb]
        prior = np.clip(prior, 1e-4, 1 - 1e-4)
        prior_logit = np.log(prior) - np.log1p(-prior)
        logit_cur = np.log(np.clip(blended, 1e-7, 1 - 1e-7)) - np.log1p(-np.clip(blended, 1e-7, 1 - 1e-7))
        blended = 1.0 / (1.0 + np.exp(-np.clip(logit_cur + 0.12 * prior_logit[None, :], -50, 50)))

        # 后处理
        for c in range(N_CLASSES):
            blended[:, c] = gaussian_filter1d(blended[:, c], sigma=0.8, mode='nearest')
        final = blended.copy()
        for t in range(1, N_WINDOWS - 1):
            final[t] = 0.5 * blended[t] + 0.25 * blended[t - 1] + 0.25 * blended[t + 1]
        final[0] = 0.6 * blended[0] + 0.4 * blended[1]
        final[-1] = 0.6 * blended[-1] + 0.4 * blended[-2]

        all_probs.append(np.clip(final, 0.0, 1.0))
        for t in range(5, 65, 5):
            all_rids.append(f"{f.stem}_{t}")
    except Exception as e:
        print(f"Skip {f.name}: {e}")

if not all_probs:
    raise ValueError("All files failed")

probs_arr = np.concatenate(all_probs, axis=0).astype(np.float32)
print(f"Done: {len(all_rids)} windows")

# ====================== 14. Submission ======================
submission = pd.DataFrame(probs_arr, columns=LABELS)
submission.insert(0, "row_id", all_rids)
submission = submission[sample_sub.columns]
submission.to_csv("/kaggle/working/submission.csv", index=False)

print(f"Saved: {submission.shape}")
assert list(submission.columns) == list(sample_sub.columns)
assert not submission.isna().any().any()
print("Format OK")

if DRYRUN:
    print("Dry-run. Commit and Submit for real score.")

BASE: /kaggle/input/competitions/birdclef-2026
PERCH: /kaggle/input/datasets/aixonvittle/perch-onnx-for-birdclef-2026/perch_v2.onnx
SED: ['sed_fold0.onnx', 'sed_fold1.onnx', 'sed_fold2.onnx', 'sed_fold3.onnx', 'sed_fold4.onnx']
SED models: 5
Mapped: 203/234 | Unmapped: 31
Genus proxy: 31 classes
Site priors: 9 | Hour: ['evening', 'morning', 'night']
Dry-run: 20 files


Perch+SED: 100%|██████████████████████████████| 20/20 [01:52<00:00,  5.65s/file]

Done: 240 windows
Saved: (240, 235)
Format OK
Dry-run. Commit and Submit for real score.
